# This notebook is for the skill predictor for Starcraft Skill

In [ ]:
# Imports
import numpy as np 
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [ ]:
# Load training data

# Read the raw CSV file into a list of lines.
with open('starcraft/train.csv') as f:
    train_lines = f.read().split('\n')
with open('starcraft/test.csv') as f:
    test_lines = f.read().split('\n')
 
# Assign a compact numeric index to each unique player name.
p = 0
playerid = {}
for line in train_lines + test_lines:
    row = line.split(',')
    if len(row) != 10:
        continue
    for name in (row[1], row[4]):
        if name not in playerid:
            playerid[name] = p
            p += 1
 
# Number of unique players and a reverse lookup for names.
nplayers = len(playerid)
playername = [''] * nplayers
for name, pid in playerid.items():
    playername[pid] = name
 
# Build win/play counts and store full game records for later use.
pKeep = 1.0   # fraction of game records to include (1.0 = keep all)
nEdge = 3     # target number of opponents per player
nKeep = 5     # maximum number of games to keep per player pair
 
nplays = np.zeros((nplayers, nplayers))
nwins = np.zeros((nplayers, nplayers))
 
# Each entry: (id_a, id_b, a_won:float, class_a:str, class_b:str, venue:str)
game_records = []
all_classes = set()
all_venues = set()
 
for line in train_lines:
    row = line.split(',')
    if len(row) != 10:
        continue
 
    # Extract both players and result fields.
    a_name, b_name = row[1], row[4]
    a_won = (row[2] == '[winner]')
    b_won = (row[5] == '[winner]')
    is_draw = a_won and b_won
    class_a = row[6].strip()
    class_b = row[7].strip()
    venue = row[9].strip()
 
    # Track all seen unit races and venues
    all_classes.update([class_a, class_b])
    all_venues.add(venue)
 
    a, b = playerid[a_name], playerid[b_name]
 
    # Filter game records by random sampling, nkeep, and nedge criteria.
    # In case of ties, credit each player with a half-win and half-loss.
    if np.random.rand() < pKeep:
        if (nplays[a, b] < nKeep) and \
           (((nplays[a, :] > 0).sum() < nEdge) or ((nplays[:, b] > 0).sum() < nEdge)):
            nplays[a, b] += 1
            nplays[b, a] += 1
            nwins[a, b] += 0.5 if is_draw else float(a_won)
            nwins[b, a] += 0.5 if is_draw else float(b_won)
            game_records.append((a, b, 0.5 if is_draw else float(a_won), class_a, class_b, venue))
 
# Build lookup tables for discrete context features.
classes = sorted(all_classes)
venues = sorted(all_venues)
class_idx = {c: i for i, c in enumerate(classes)}
venue_idx = {v: i for i, v in enumerate(venues)}
NC, NV = len(classes), len(venues)
 
print("Training data:")
print(f"Players : {nplayers}")
print(f"Classes : {classes}")
print(f"Venues  : {venues}")
print(f"Games   : {len(game_records)}")

# Process test data in the same way, but only keep player indices and outcomes.
test_records = []
for line in test_lines:
    row = line.split(',')
    if len(row) != 10:
        continue
    a_name, b_name = row[1], row[4]
    a_won = row[2] == '[winner]'
    b_won = row[5] == '[winner]'
    is_draw = a_won and b_won
    a, b = playerid[a_name], playerid[b_name]
    test_records.append((a, b, 0.5 if is_draw else float(a_won)))


print("\nTesting data:")
print(f"Players seen in test : {len(set(a for a,b,_ in test_records) | set(b for a,b,_ in test_records))}")
print(f"Games                : {len(test_records)}")

# Installments
hots = heart of the swarm (Z)

lotv = legacy of the void (P) ( newest and which most people play on )

wol = wings of liberty (T) ( oldest )
# Classes
Protoss (P)

Terran (T)

Zerg (Z)

(07/17/2016,MC,[loser],0–1,Cure,[winner],P,T,LotV,offline)

(Example entry)

In [ ]:
# Latent Variable Skill Model
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))


class LatentSkillModel:
    """
    Maintains a Gaussian posterior N(mu_i, sigma_i^2) per player.
    P(i beats j) = sigmoid(mu_i - mu_j)
    Updates are a Laplace approximation:
      - mu updated via gradient of log-likelihood
      - sigma updated via observed Fisher information
    """

    def __init__(self, nplayers, prior_mu=0.0, prior_sigma=1.0, lr=0.01):
        self.nplayers = nplayers
        self.prior_mu = prior_mu
        self.prior_sigma = prior_sigma
        self.lr = lr
        self.mu = np.full(nplayers, prior_mu, dtype=float)
        self.sigma = np.full(nplayers, prior_sigma, dtype=float)
        self.n_games = np.zeros(nplayers, dtype=int)

    def update(self, a, b, a_won):
        """Single-game posterior update for players a and b."""
        p_a = sigmoid(self.mu[a] - self.mu[b])
        err = a_won - p_a
        fisher = p_a * (1.0 - p_a)

        # Update mu using gradient of log-likelihood.
        self.mu[a] += self.lr * self.sigma[a]**2 * err
        self.mu[b] -= self.lr * self.sigma[b]**2 * err

        # Update sigma using observed Fisher information.
        self.sigma[a] = 1.0 / np.sqrt(1.0 / self.sigma[a]**2 + fisher)
        self.sigma[b] = 1.0 / np.sqrt(1.0 / self.sigma[b]**2 + fisher)

        # Update game count.
        self.n_games[a] += 1
        self.n_games[b] += 1

    def fit(self, game_records, n_passes=5, print_every=5, tol=1e-4):
        """Run n_passes passes of updates over the game records or until delta reaches tol."""
        self.mu[:] = self.prior_mu
        self.sigma[:] = self.prior_sigma
        self.n_games[:] = 0
        for pass_num in range(n_passes):
            mu_before = self.mu.copy()
            #Ignore categorical features for now, can implement later
            for a, b, a_won, *_ in game_records:
                self.update(a, b, a_won)
            delta = np.abs(self.mu - mu_before).mean()
            if (pass_num + 1) % print_every == 0:
                print(f"Pass {pass_num+1}: mean |Δμ| = {delta:.6f}")
            if delta < tol:
                print(f"Converged at pass {pass_num+1}.")
                break

    def predict(self, a, b):
        """P(a beats b) under current posterior means."""
        return sigmoid(self.mu[a] - self.mu[b])

    def top_k(self, k=20):
        """Print the top k players by posterior mean skill."""
        ranked = np.argsort(self.mu)[::-1]
        print(f"\n{'Rank':<5} {'Player':<22} {'μ':>8} {'σ':>8} {'Games':>6}")
        print("-" * 55)
        for rank, pid in enumerate(ranked[:k], 1):
            print(f"{rank:<5} {playername[pid]:<22} {self.mu[pid]:>8.4f} "
                  f"{self.sigma[pid]:>8.4f} {self.n_games[pid]:>6}")

In [ ]:
model = LatentSkillModel(nplayers, lr=0.01)
model.fit(game_records, n_passes=200, print_every=10, tol=1e-4)
model.top_k(k=20)

## Evaluation ##

Here we evaluate the model's performance on the testing data based off at raw accuracy and also log-likelihood. We only test on games where both players are in the training set. This filtering was done when we loaded the testing data alongside the training data.

In [ ]:
correct = 0
total = 0
log_loss = 0.0
eps = 1e-7

for a, b, a_won in test_records:
    p_a = model.predict(a, b)
    p_a_clipped = np.clip(p_a, eps, 1 - eps)
    
    correct += int((p_a > 0.5) == (a_won > 0.5))
    log_loss -= a_won * np.log(p_a_clipped) + (1 - a_won) * np.log(1 - p_a_clipped)
    total += 1

print(f"Test accuracy : {correct}/{total} = {correct/total:.4f}")
print(f"Test log-loss : {log_loss/total:.4f}  (random baseline = {np.log(2):.4f})")

# Sources: